# Analisi dei Risultati dell'Addestramento DNN

Questo notebook presenta l'analisi completa dei risultati ottenuti dal training di diverse configurazioni di reti neurali profonde. Il focus principale è la valutazione dell'impatto degli iperparametri (Learning Rate, Funzione di Attivazione, Dropout e Ottimizzatore) sulle performance finali del modello, misurate principalmente tramite la `Validation Accuracy`.

## 1. Distribuzione delle Performance per Iperparametro
![Boxplots](plots/1_hyperparams_boxplots.png)

### Analisi dei Comportamenti Sistematici e Outliers:
Il grafico a boxplot offre una visione immediata e complessiva degli effetti marginali di ciascun iperparametro:

- **Funzione di Attivazione (Comportamento Sistematico di Degradazione):** Il risultato più netto dell'intero studio è evidente guardando il pannello in alto a destra. Si denota **un comportamento sistematico in cui la funzione `sigmoid` fornisce risultati estremamente cattivi e inaffidabili** se comparata alle moderne controparti. Questa funzione soffre tipicamente del problema del *vanishing gradient*, che impedisce alla rete di apprendere in modo efficace, portando a mediane molto basse e un'alta varianza di risultati deboli (molti modelli restano confinati in accuracy poco sopra il caso random). Viceversa, **`relu` ed `elu` si dimostrano scelte eccellenti e sistematicamente vincenti**, esibendo non solo mediane elevate, ma una nube di dispersione concentrata nei valori superiori.
- **Learning Rate:** Le distribuzioni in funzione del LR mostrano la tipica forma a "campana sfalsata". Valori intermedi ($10^{-2}$) ospitano la quasi totalità dei modelli top-performing. Valori troppo bassi ($10^{-5}$, $10^{-4}$) spesso faticano a convergere (mediana ed efficienza inferiore con molti outliers al ribasso), mentre valori come $10^{-3}$, $10^{-1}$ presentano numerosi **comportamenti outliers** sotto forma di crolli vertiginosi in performance, causati da gradienti instabili o logiche divergenti.
- **Dropout:** Il Dropout impatta in maniera meno drastica sulle distribuzioni globali in questo search space circoscritto. Asd ongi modo si vedono risultati leggermente migliori per dropout minori di 0.2.
- **Ottimizzatore:** Tra gli ottimizzatori, si vedono lievi differenze in valore mediano, ma tutti contengono outliers molto bassi e molto alti, spesso corrispondenti alle configurazioni maligne (es. combinate con sigmoid o LR sballati).

## 2. Sensibilità al Learning Rate rispetto all'Ottimizzatore
![Val Acc per Optimizer](plots/2_val_acc_by_lr_optimizer.png)

### Motivazione delle Scelte (Ottimizzatore):
Questo interaction plot evidenzia la relazione dinamica tra ottimizzatore e LR:
- Metodi con update adattivi o con *momentum* come `Adam` e `RMSprop`, tendono ad avere uno *sweet-spot* iper-performante nei range centrali ($10^{-4}$ - $10^{-3}$).
- **Outliers e Instabilità Numerica:** Fintanto che il LR si mantiene su ordini conservativi per la categoria, i modelli mostrano un comportamento omogeneo. All'avvicinarsi a valori di LR esplosivi ed inappropriati ($10^{-1}$), osserviamo un collasso drastico sistematico su quasi tutti gli optimizers: queste metriche di accuratezza precipitate configurano dei veri e propri "outliers di sistema". Per LR esageratamente alti (0.5) è sistematico per tutti i modelli un drastico calo delle prestazioni.
**La scelta ottimale impone l'adozione forte di un LR nell'intorno di $10^{-2}$**.
Si noti come per nesterov outliers portino a risultati soprendentemente buoni ad alti LR.

## 3. Sensibilità al Learning Rate rispetto all'Attivazione
![Val Acc per Activation](plots/3_val_acc_by_lr_activation.png)

### Riconferma dei Comportamenti Sistematici Estremi:
Il grafico ripropone quanto osservato in precedenza conferendogli robustezza parametrica:
- La traccia corrispondente alla **`sigmoid` è la peggiore quasi costantemente** ad ogni valore di learning rate indagato. Raggiunge a stento o lambisce le performance marginali delle altre attivazioni negli spot isolati dove `relu` e `elu` stanno già cronicamente collassando per instabilità a regimi elevati (LR = $10^{-1}$).
- I profili di convergenza scalare di **`relu` e `elu` descrivono una forma parabolica chiara in log-scale**, dettando come gold standard empirico la loro rapida implementazione per la stragrande maggioranza delle task di reti feed-forward. Mantenere l'attivazione `relu`/`elu` è un prerequisito fondamentale in questo ecosistema per innalzare il benchmark massimo di accuratezza.

## 4. Analisi delle Top 5 Configurazioni
![Top 5 Models](plots/4_top_5_models.png)

### Motivazione dei Vincitori Eccellenti:
Il confronto ravvicinato tra i primi 5 in assoluto nella Validation Accuracy con la metrica di hold-out (Test Accuracy):
- **Scelte vincenti:** Come atteso, essun run best-in-class usa sigmoid o LR fuori da $[10^{-3};10^{-1}]$. Tutte si allineano con assoluta dominanza su attivazioni `elu` o `relu` appaiate a LR perfettamente centrate.
- **Generalizzazione Robusta (Assenza di Overfitting):** Le configurazioni d'élite registrano un divario insignificante, in alcuni limitrofi quasi invisibile, tra Validation e Test Accuracy. Questo indica che la topologia applicata (incorniciante il fattore di dropout esatto di 0.1/0.2 riscontrabile) regolarizza ottimamente la rete. In altre parole la prestazione validativa del modello scala solidamente senza deviazioni irrazionali sulla generalizzazione invisibile; nessuno di questi risultati deve essere etichettato come benefico *statistical outlier*.

## 5. Local Sensitivity Analysis (I 5 Modelli Top)
Nelle sottosezioni sottostanti alteriamo la ricetta aurea per ciascuno dei top 5 modelli scostando, *ceteris paribus* (a parità di tutti gli altri fattori), un attore incognito per volta (partendo e confrontando dal setup ottimale indicato tramite una stella rossa sul grafico).

### Top 1 Model Analysis
![Top 1 Analysis](plots/5_top1_model_analysis.png)

### Top 2 Model Analysis
![Top 2 Analysis](plots/5_top2_model_analysis.png)

### Top 3 Model Analysis
![Top 3 Analysis](plots/5_top3_model_analysis.png)

### Top 4 Model Analysis
![Top 4 Analysis](plots/5_top4_model_analysis.png)

### Top 5 Model Analysis
![Top 5 Analysis](plots/5_top5_model_analysis.png)

### Esplorazioni di Neighborhood (Robustezza e Cadute):
- **Reattività Sistematica all'Attivazione:** Se da uno qualsiasi di questi 5 capolavori parametrici si alterasse unicamente il parametro *activation* in direzione della deprecata `sigmoid` (congelando perfino il LR ideale all'eccellenza), si verificherebbe l'ennesimo collasso precipitoso e drastico delle capability validatrici della rete neurale. La sigmoide è la variabile latente di deterioramento incontrastabile delle performance riscontrate.
- **Sensibilità Parametrica Punti-Chiave:** In molti di tali slice (sezione del gradiente di variazione ad un solo grado di libertà), le varianze irruente di `optimizer` prescelto o di `learning_rate` generano precipitosi scarti negativi di validation accuracy originando plateali *outliers* della deviazione da ottimalità.
- La lezione chiave evidenziata ribadisce che il successo empirico di un network profondo risiede in uno stretto *sweet-spot* o dominio unificato: perturbare scioccamente una sinergia calibrata di un Top Model (cambiando l'optimizer a favore magari dell'insidioso e primitivo SGD su quel range) provocherà una decrescita drastica. Le scelte migliori esposte dimostrano l'inviolabilità del trade-off calibrato fra una funzione non bloccante come **relu/elu** unita as un modulo dinamico ottimale come **Adam/RMSprop** con un Rate di convergenza centrale.